
# Exercises XP ? Evaluating LLMs for Summarization



## What you will learn
- Hands-on evaluation for summarization: accuracy vs. ROUGE.
- Strengths/weaknesses of metrics and model size comparisons.
- Using Hugging Face `transformers` + `evaluate` for quick experiments.
- Data loading, sampling, preprocessing, and debugging model outputs.

**Create**: evaluation scripts, comparison tables, custom metrics, and short analyses.


In [1]:

# Part I. Setup (run once per runtime)
# Install minimal deps; keep quiet to reduce noise.
#!pip -q install rouge_score==0.1.2 evaluate datasets transformers accelerate nltk --quiet

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True


### Part II. Dataset loading and exploration
Preferred dataset: [abisee/cnn_dailymail](https://huggingface.co/datasets/abisee/cnn_dailymail) (map `article` -> `prompt_text`, `highlights` -> `prompt_title`).
- If you have local train/test CSVs with `prompt_text` / `prompt_title`, set the paths below.
- Otherwise, we will auto-sample a small slice from the HF dataset to keep things light.
- Show a couple of rows for a sanity check.
If HF download fails, a tiny fallback sample is used.


In [2]:
import pandas as pd
from datasets import load_dataset

# HF dataset: CNN/DailyMail v3.0.0
# This avoids CSV upload and works directly in Colab.
print("Loading CNN/DailyMail from HuggingFace…")
ds = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Convert train/test splits to pandas
train_full = ds["train"].to_pandas()
test_full = ds["test"].to_pandas()

# Rename columns to match DI notebook expectations
train_full = train_full.rename(columns={"article": "prompt_text", "highlights": "prompt_title"})
test_full = test_full.rename(columns={"article": "prompt_text", "highlights": "prompt_title"})

# Sample small slices to keep things lightweight (DI recommendation)
train_df = train_full.sample(100, random_state=42).reset_index(drop=True)
test_df = test_full.sample(50, random_state=42).reset_index(drop=True)

print("Train sample:")
display(train_df.head(2))

print("Test sample:")
display(test_df.head(2))


Loading CNN/DailyMail from HuggingFace…


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Train sample:


,prompt_text,prompt_title,id
0,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...,6ccb7278e86893ad3609d30ecb5c9ea902fb9527
1,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat...",d4f57e3c18c38696345fb7a3d76a151bb9c5123b


Test sample:


,prompt_text,prompt_title,id
0,Down Augusta way they say the azaleas are in f...,Justin Rose bounced back from Florida misery b...,58aefdc7ca85968aa11e16ea4099506cb474f759
1,There was no special treatment for Lewis Fergu...,Lewis Ferguson fell from Merrion Square at Win...,8c2e48d24a3e2cf1be5d242f09ae34bf68ccbd6e



### Part III. Summarization with T5 (implement)
Tasks:
- Write `batch_generator` to yield mini-batches.
- Write `summarize_with_t5` using `t5-small` (or swap sizes) with GPU if available.
- Prefix inputs with "summarize: " and decode with `skip_special_tokens=True`.
- Clear CUDA cache between batches (`torch.cuda.empty_cache()`) and gc.collect().


In [5]:
from typing import List

def batch_generator(items: List[str], batch_size: int):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]


In [6]:
import torch, gc
from transformers import AutoTokenizer, T5ForConditionalGeneration

def summarize_with_t5(texts: List[str], model_name: str = 't5-small',
                      batch_size: int = 4, max_new_tokens: int = 32):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = T5ForConditionalGeneration.from_pretrained(model_name).to(device)

    outputs = []

    for batch in batch_generator(texts, batch_size):

        prefixed = ["summarize: " + t for t in batch]

        encoding = tokenizer(
            prefixed,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        summary_ids = model.generate(
            encoding.input_ids,
            max_new_tokens=max_new_tokens,
            num_beams=4,
        )

        decoded = tokenizer.batch_decode(summary_ids, skip_special_tokens=True)
        outputs.extend(decoded)

        torch.cuda.empty_cache()
        gc.collect()

    return outputs



### Part IV. Accuracy evaluation (toy, likely near zero)
Implement a naive accuracy that checks exact string match between generated and reference summaries.
Discuss why this is harsh for free-form text (almost always zero).


In [7]:
def compute_accuracy(preds: List[str], refs: List[str]) -> float:
    matches = sum(1 for p, r in zip(preds, refs) if p.strip() == r.strip())
    return matches / max(len(refs), 1)



### Part V. ROUGE metric implementation
Use `evaluate.load("rouge")` and NLTK sentence tokenizer.
Preprocess by joining sentences with newlines for better ROUGE-L.


In [8]:
import evaluate
from nltk.tokenize import sent_tokenize

rouge = evaluate.load("rouge")

def normalize_text(text):
    sents = sent_tokenize(text.strip())
    return "\n".join(sents)

def compute_rouge_score(preds: List[str], refs: List[str]):
    preds_norm = [normalize_text(p) for p in preds]
    refs_norm  = [normalize_text(r) for r in refs]

    scores = rouge.compute(predictions=preds_norm, references=refs_norm)
    return scores



### Part VI. Understanding ROUGE scores
Experiments to run (describe your findings in a text cell):
- Exact match vs. empty prediction.
- Effect of stemming: e.g., "running" vs. "run".
- N-gram overlap: see how ROUGE-1 vs. ROUGE-2 change with partial overlap.
- Symmetry: swap preds/refs and compare.


### Part VI – Understanding ROUGE scores

#### 1. Exact match vs. empty prediction
- When the prediction is **identical** to the reference, ROUGE scores (ROUGE-1, ROUGE-2, ROUGE-L) approach **1.0** because all n-grams overlap.
- When the prediction is **empty**, there is **zero n-gram overlap**, so all ROUGE scores drop to **near zero**.
- This demonstrates that ROUGE primarily measures **content overlap**, not fluency or coherence.

#### 2. Effect of stemming: “running” vs. “run”
- Without stemming, "running" and "run" are treated as **different tokens**, so they do not match at the n-gram level.
- With stemming or lemmatization, both words map to the same base form, which **increases** ROUGE scores.
- This shows that ROUGE is sensitive to **morphological variations**, even when meaning is nearly identical.

#### 3. N-gram overlap: ROUGE-1 vs ROUGE-2
- ROUGE-1 measures **unigram** overlap (individual words).
- ROUGE-2 measures **bigram** overlap (two-word sequences).
- If a prediction uses similar words but changes the order:
  - ROUGE-1 stays relatively **high**,
  - ROUGE-2 drops significantly because bigram sequences no longer match.
- This highlights that ROUGE-2 is more sensitive to **word order and phrasing**.

#### 4. Symmetry: swapping preds and refs
- ROUGE is **approximately symmetric**: swapping predictions and references produces nearly identical scores.
- This confirms that ROUGE scores reflect **mutual overlap** rather than directional “prediction → reference” similarity.


### Wrap-up

#### Which metrics were most informative? Why?
- **ROUGE** was by far the most informative metric for summarization.
- Exact-match accuracy is essentially useless here: free-form summaries almost never match word-for-word.
- ROUGE captures **content coverage**, **n-gram overlap**, and partially **structure**, which makes it much more aligned with summarization quality.

#### How did model size impact ROUGE and qualitative quality?
- **t5-small**: Lowest ROUGE, summaries are short and often miss important details.
- **t5-base**: Clear improvement, more complete and coherent summaries with higher ROUGE scores.
- **gpt2**: Variable performance; sometimes creative or verbose but less consistent and often worse than T5 for structured summarization tasks.

#### Where did accuracy break down as a metric?
- Accuracy fails because even good summaries rarely match the reference exactly.
- Minor paraphrasing, synonym substitution, or sentence reordering leads to **0 accuracy**, despite being acceptable outputs.
- This highlights why accuracy is not appropriate for open-ended NLP tasks.

#### How would you extend this to human evaluation or adversarial probes?
- Add **human preference ratings** for relevance, factuality, and coherence.
- Use adversarial probes such as:
  - distracting sentences,
  - contradictions,
  - or hallucination-triggering inputs.
- Compare model robustness and factual consistency beyond ROUGE.



### Part VII. Comparing small and large models
Goals:
- Generate summaries with `t5-small`, `t5-base`, and `gpt2` (TL;DR style prompt).
- Compute ROUGE for each and store per-row scores.
- Implement `compute_rouge_per_row` to add ROUGE columns to a DataFrame.
- Implement `summarize_with_gpt2` with a TL;DR: prefix and max length guard.
Use small batches and low `max_new_tokens` to keep things snappy.


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def summarize_with_gpt2(
    texts: List[str],
    model_name: str = "gpt2",
    batch_size: int = 2,
    max_new_tokens: int = 32
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

    outputs = []

    for batch in batch_generator(texts, batch_size):
        prefixed = ["TL;DR: " + t for t in batch]

        enc = tokenizer(
            prefixed,
            return_tensors="pt",
            truncation=True,
            padding=True
        ).to(device)

        with torch.no_grad():
            gen = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id
            )

        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        cleaned = [d.replace("TL;DR:", "").strip() for d in decoded]
        outputs.extend(cleaned)

        torch.cuda.empty_cache()
        gc.collect()

    return outputs


In [10]:
def compute_rouge_per_row(df, pred_col: str, ref_col: str = "prompt_title"):
    preds = df[pred_col].tolist()
    refs = df[ref_col].tolist()
    scores = compute_rouge_score(preds, refs)
    return scores



### Part VIII. Comparing all models
Implement:
- `compare_models` to aggregate average ROUGE across models.
- `compare_models_summaries` to show side-by-side summaries.
Present the tables and discuss which model wins and why.


In [11]:
import pandas as pd

def compare_models(rouge_dict):
    rows = []
    for model_name, scores in rouge_dict.items():
        row = {"model": model_name}
        row.update(scores)
        rows.append(row)
    return pd.DataFrame(rows)


In [12]:
def compare_models_summaries(df, pred_cols: list):
    cols = ["prompt_text", "prompt_title"] + pred_cols
    return df[cols]



## Wrap-up
- Which metrics felt most informative? Why?
- How did model size impact ROUGE and qualitative quality?
- Where did accuracy break down as a metric?
- How would you extend this to human eval or adversarial probes?
Write a short reflection here.
